# Explanation-Supervised Attention CNN — NIH ChestX-ray14
## Google Colab Training Notebook (Full Pipeline)

---

### Pre-flight checklist — do these BEFORE running any cell

| # | What | How |
|---|------|-----|
| 1 | **GPU runtime** | Runtime → Change runtime type → **T4 GPU** |
| 2 | **Kaggle API key** | kaggle.com → Profile → Settings → API → **Create New Token** |
| 3 | **Project code in Drive** | Upload `dl-project-code.zip` to `My Drive/` root |

### What this notebook does
- Cells 1–5: Environment setup (~15–35 min, one-time per session)
- Cells 6–8: Data loading, splitting, DataLoader sanity check
- Phase 2 (Cells 9–11): Train 3 baselines — ResNet50, DenseNet121, EfficientNet-B0
- Phase 3 (Cells 12–13): Train attention variants — ResNet50 + DenseNet121
- Phase 4 (Cell 14): Ablations — remove L_attn / L_corr separately
- Cells 15–18: Full evaluation, report tables, figures

### Storage strategy
- **Dataset** → local Colab disk `/content/nih-chest-xrays/` — 6 batches ≈ 54 K images, ~21 GB
- **Outputs** (checkpoints, logs, figures) → Google Drive (a few hundred MB total)
- Each new session: re-run Cells 1–8 to restore env, training resumes from Drive checkpoints

---

In [6]:
import shutil, os
shutil.rmtree('/content/nih-chest-xrays', ignore_errors=True)
print('Cleared.'); os.system('df -h /content')

Cleared.


0

In [14]:
!pip install --upgrade kaggle -q

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 132.8/132.8 kB 6.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 230.0/230.0 kB 14.8 MB/s eta 0:00:00


In [7]:
# Cell 1 — Install dependencies + verify GPU
import subprocess, sys

print('Installing packages...')
subprocess.run(
    [sys.executable, '-m', 'pip', 'install',
     'kaggle', 'timm', 'albumentations', 'scikit-learn', 'seaborn', '-q'],
    check=True
)
print('Packages installed.')

import torch
print(f'PyTorch  : {torch.__version__}')
print(f'CUDA     : {torch.cuda.is_available()}')
if torch.cuda.is_available():
    print(f'GPU      : {torch.cuda.get_device_name(0)}')
    print(f'Mem      : {torch.cuda.get_device_properties(0).total_memory/1e9:.1f} GB')
else:
    print('NO GPU — go to Runtime → Change runtime type → T4 GPU, then restart.')


Installing packages...
Packages installed.
PyTorch  : 2.11.0+cu128
CUDA     : True
GPU      : Tesla T4
Mem      : 15.6 GB


In [23]:
# Cell 2 — Kaggle credentials (from Colab Secrets)
import os
from google.colab import userdata

KAGGLE_TOKEN = userdata.get('KAGGLE_API_TOKEN')

os.makedirs('/root/.kaggle', exist_ok=True)
with open('/root/.kaggle/access_token', 'w') as f:
    f.write(KAGGLE_TOKEN)
os.chmod('/root/.kaggle/access_token', 0o600)
os.environ['KAGGLE_API_TOKEN'] = KAGGLE_TOKEN
print('Kaggle credentials ready.')

Kaggle credentials ready.


In [24]:
# Cell 3 — Load project code
# Looks for dl-project-code.zip in Google Drive first;
# if not there, prompts direct upload.
import sys, os, zipfile
from google.colab import files as colab_files

LOCAL_CODE = '/content/dl-project'
DRIVE_ZIP  = '/content/drive/MyDrive/dl-project-code.zip'

def _extract(zip_path, dest):
    os.makedirs(dest, exist_ok=True)
    with zipfile.ZipFile(zip_path, 'r') as z:
        z.extractall(dest)
    contents = os.listdir(dest)
    if len(contents) == 1 and os.path.isdir(os.path.join(dest, contents[0])):
        return os.path.join(dest, contents[0])
    return dest

if os.path.isdir(LOCAL_CODE) and os.path.isdir(os.path.join(LOCAL_CODE, 'src')):
    print('Project code already extracted.')
elif os.path.exists(DRIVE_ZIP):
    print('Found dl-project-code.zip in Drive — extracting...')
    LOCAL_CODE = _extract(DRIVE_ZIP, LOCAL_CODE)
else:
    print('Not found in Drive. Upload dl-project-code.zip now:')
    uploaded   = colab_files.upload()
    zip_name   = list(uploaded.keys())[0]
    LOCAL_CODE = _extract(zip_name, LOCAL_CODE)

if LOCAL_CODE not in sys.path:
    sys.path.insert(0, LOCAL_CODE)

print(f'Project root : {LOCAL_CODE}')
print('Contents     :', sorted(os.listdir(LOCAL_CODE)))
try:
    import src; print('src importable: OK')
except ImportError as e:
    print(f'ERROR: src not importable — {e}')


Not found in Drive. Upload dl-project-code.zip now:


Saving dl-project-code.zip to dl-project-code.zip
Project root : /content/dl-project/dl-project-code
Contents     : ['config.yaml', 'notebooks', 'src']
src importable: OK


In [25]:
# Cell 4 — Mount Google Drive (outputs only)
import os, shutil
from google.colab import drive
drive.mount('/content/drive', force_remount=False)

DRIVE_OUTPUTS  = '/content/drive/MyDrive/DL_Project_Outputs'
CHECKPOINT_DIR = os.path.join(DRIVE_OUTPUTS, 'checkpoints')
LOG_DIR        = os.path.join(DRIVE_OUTPUTS, 'logs')
FIGURE_DIR     = os.path.join(DRIVE_OUTPUTS, 'figures')

for d in [CHECKPOINT_DIR, LOG_DIR, FIGURE_DIR]:
    os.makedirs(d, exist_ok=True)

_, _, free = shutil.disk_usage('/content/drive/MyDrive')
print(f'Drive mounted. Free space: {free/1e9:.1f} GB')
print(f'Outputs → {DRIVE_OUTPUTS}')
if free < 0.5e9:
    print('WARNING: Less than 0.5 GB free in Drive — checkpoints may fail!')


Mounted at /content/drive
Drive mounted. Free space: 5.3 GB
Outputs → /content/drive/MyDrive/DL_Project_Outputs


In [28]:
# Cell 5 — Download + extract FULL NIH ChestX-ray14 (112K images)
import os, zipfile

LOCAL_DATASET = '/content/nih-chest-xrays'
IMG_DIR       = os.path.join(LOCAL_DATASET, 'images')
os.makedirs(IMG_DIR, exist_ok=True)

def find_zip():
    for f in os.listdir(LOCAL_DATASET):
        if f.endswith('.zip'):
            return os.path.join(LOCAL_DATASET, f)
    return None

# Download
zip_path = find_zip()
if zip_path:
    print(f'Zip already present: {zip_path}')
else:
    print('Downloading full dataset (~42 GB). Do NOT close tab...')
    !kaggle datasets download -d nih-chest-xrays/data -p "{LOCAL_DATASET}"
    zip_path = find_zip()

print(f'Zip: {os.path.getsize(zip_path)/1e9:.1f} GB')

# Inspect
with zipfile.ZipFile(zip_path) as z:
    all_entries = z.namelist()
img_entries = [f for f in all_entries if f.lower().endswith(('.png','.jpg','.jpeg'))]
csv_entries = [f for f in all_entries if f.endswith('.csv')]
print(f'Images in zip : {len(img_entries):,}')
print(f'CSVs          : {csv_entries}')

# Extract CSVs
print('\nExtracting CSVs...')
with zipfile.ZipFile(zip_path) as z:
    for csv_f in csv_entries:
        dest = os.path.join(LOCAL_DATASET, os.path.basename(csv_f))
        if not os.path.exists(dest):
            z.extract(csv_f, LOCAL_DATASET)
            extracted = os.path.join(LOCAL_DATASET, csv_f)
            if os.path.exists(extracted) and extracted != dest:
                import shutil; shutil.move(extracted, dest)
        print(f'  ✓ {os.path.basename(csv_f)}')

# Extract ALL images
already = len(os.listdir(IMG_DIR))
remaining = [f for f in img_entries
             if not os.path.exists(os.path.join(IMG_DIR, os.path.basename(f)))]
print(f'\nImages already extracted : {already:,}')
print(f'Images to extract        : {len(remaining):,}')

with zipfile.ZipFile(zip_path) as z:
    for i, img_f in enumerate(remaining):
        dest = os.path.join(IMG_DIR, os.path.basename(img_f))
        data = z.read(img_f)
        with open(dest, 'wb') as out:
            out.write(data)
        if (i+1) % 10000 == 0:
            print(f'  {already+i+1:,} / {len(img_entries):,} extracted...')

# Delete zip
print(f'\nDeleting zip ({os.path.getsize(zip_path)/1e9:.1f} GB)...')
os.remove(zip_path)

!df -h /content
n = len(os.listdir(IMG_DIR))
print(f'\nDone — {n:,} images ready.')

Dataset URL: https://www.kaggle.com/datasets/nih-chest-xrays/data
License(s): CC0-1.0
100% 42.0G/42.0G [08:17<00:00, 90.7MB/s]

Zip: 45.1 GB
Images in zip : 112,120
CSVs          : ['BBox_List_2017.csv', 'Data_Entry_2017.csv']

Extracting CSVs...
  ✓ BBox_List_2017.csv
  ✓ Data_Entry_2017.csv

Images already extracted : 0
Images to extract        : 112,120
  10,000 / 112,120 extracted...
  20,000 / 112,120 extracted...
  30,000 / 112,120 extracted...
  40,000 / 112,120 extracted...
  50,000 / 112,120 extracted...
  60,000 / 112,120 extracted...
  70,000 / 112,120 extracted...
  80,000 / 112,120 extracted...
  90,000 / 112,120 extracted...
  100,000 / 112,120 extracted...
  110,000 / 112,120 extracted...

Deleting zip (45.1 GB)...
Filesystem      Size  Used Avail Use% Mounted on
overlay         236G   90G  147G  38% /

Done — 112,120 images ready.


In [33]:
# Cell 6 — Configuration
import os

cfg = {
    # Paths
    'data_dir'       : LOCAL_DATASET,
    'sample_dir'     : os.path.join(LOCAL_DATASET, 'images'),
    'csv_entry'      : 'Data_Entry_2017.csv',
    'csv_bbox'       : 'BBox_List_2017.csv',
    'output_dir'     : DRIVE_OUTPUTS,
    'checkpoint_dir' : CHECKPOINT_DIR,
    'log_dir'        : LOG_DIR,
    'figure_dir'     : FIGURE_DIR,

    # Dataset
    'image_size'  : 224,
    'num_classes' : 14,
    'subset_size' : 30000,   # None = use all ~54K downloaded images

    # Split
    'val_frac'    : 0.10,
    'test_frac'   : 0.10,
    'random_seed' : 42,

    # Runtime
    'device'         : 'cuda',
    'num_workers'    : 2,
    'mixed_precision': True,
    'sample_mode'    : False,

    # Model
    'backbone'             : 'resnet50',
    'pretrained'           : True,
    'attention_resolution' : 7,
    'use_channel_attn'     : True,

    # Loss
    'lambda1'        : 1.0,
    'lambda2'        : 0.5,
    'focal_gamma'    : 2.0,
    'dice_weight'    : 1.0,
    'mse_weight'     : 0.5,
    'sparsity_weight': 0.01,

    # Training
    'epochs'         : 10,
    'batch_size'     : 64,
    'lr'             : 1e-4,
    'weight_decay'   : 1e-5,
    'scheduler'      : 'cosine',
    'warmup_epochs'  : 2,
    'grad_clip'      : 1.0,

    # Logging
    'log_interval'       : 100,
    'val_interval'       : 1,
    'checkpoint_metric'  : 'macro_auc',
    'early_stop_patience': 5,
}

print('Config ready.')
print(f'  Data       : {cfg["data_dir"]}')
print(f'  Images     : {len(os.listdir(cfg["sample_dir"])):,}')
print(f'  Outputs    : {cfg["output_dir"]}')
print(f'  Device     : {cfg["device"]}')
print(f'  Epochs     : {cfg["epochs"]}  |  Batch: {cfg["batch_size"]}  |  LR: {cfg["lr"]}')


Config ready.
  Data       : /content/nih-chest-xrays
  Images     : 112,120
  Outputs    : /content/drive/MyDrive/DL_Project_Outputs
  Device     : cuda
  Epochs     : 10  |  Batch: 64  |  LR: 0.0001


In [34]:
# Cell 7 — Load data + patient-level split
from src.data.splits import (
    load_dataframe, build_balanced_subset,
    patient_level_split, get_class_weights,
    compute_cooccurrence_matrix, CLASS_NAMES
)

df = load_dataframe(cfg['data_dir'])
print(f'CSV rows        : {len(df):,}')
print(f'Unique patients : {df["patient_id"].nunique():,}')

if cfg.get('subset_size'):
    df = build_balanced_subset(df, cfg['subset_size'], cfg['random_seed'])
    print(f'Balanced subset : {len(df):,}')

train_df, val_df, test_df = patient_level_split(
    df, cfg['val_frac'], cfg['test_frac'], cfg['random_seed']
)
print(f'\nTrain: {len(train_df):,}  Val: {len(val_df):,}  Test: {len(test_df):,}')

pos_weights = get_class_weights(train_df)
cooc        = compute_cooccurrence_matrix(train_df)
print(f'Co-occurrence matrix: {cooc.shape}')
print(f'Classes: {CLASS_NAMES}')


CSV rows        : 112,120
Unique patients : 30,805
Balanced subset : 23,890
[splits] ✓ No patient-level leakage detected.
[splits] Patients → train:7,438  val:930  test:930
[splits] Images   → train:19,204  val:2,248  test:2,438
[splits] Boxed images → train:705  val:96  test:79

Train: 19,204  Val: 2,248  Test: 2,438
Co-occurrence matrix: (14, 14)
Classes: ['Atelectasis', 'Cardiomegaly', 'Effusion', 'Infiltration', 'Mass', 'Nodule', 'Pneumonia', 'Pneumothorax', 'Consolidation', 'Edema', 'Emphysema', 'Fibrosis', 'Pleural_Thickening', 'Hernia']


In [35]:
# Cell 8 — Build DataLoaders + sanity check
from src.data.dataset import build_dataloaders

train_loader, val_loader, test_loader = build_dataloaders(
    cfg, train_df, val_df, test_df
)

imgs, labels, masks, has_box = next(iter(train_loader))
print('Batch shapes:')
print(f'  images  : {tuple(imgs.shape)}')
print(f'  labels  : {tuple(labels.shape)}')
print(f'  masks   : {tuple(masks.shape)}')
print(f'  has_box : {has_box.tolist()[:6]}')
print(f'\nTrain batches : {len(train_loader):,}')
print(f'Val batches   : {len(val_loader):,}')
print(f'Test batches  : {len(test_loader):,}')
print('DataLoaders OK.')


[dataset] train=19,204  val=2,248  test=2,438
Batch shapes:
  images  : (64, 3, 224, 224)
  labels  : (64, 14)
  masks   : (64, 7, 7)
  has_box : [0, 0, 0, 0, 1, 0]

Train batches : 300
Val batches   : 36
Test batches  : 39
DataLoaders OK.


---
## Phase 2 — Baseline Models
Three standard CNN classifiers, no attention module.
Expected ~2-4 h per model on T4 GPU (30 epochs, ~54K images).

> **If the session disconnects**: re-run Cells 1–8, then only the cells for unfinished models. Checkpoints on Drive are safe.

In [32]:
# Cell 9 — Phase 2: Train ResNet50 Baseline
from src.train import train

cfg['backbone'] = 'resnet50'
print('Training ResNet50 baseline...')
history_r50_base = train(cfg, variant=False)
best = max(h['macro_auc'] for h in history_r50_base['val'])
print(f'ResNet50 baseline done. Best val macro AUC: {best:.4f}')


Training ResNet50 baseline...
[train] Loading data...
[splits] ✓ No patient-level leakage detected.
[splits] Patients → train:24,643  val:3,081  test:3,081
[splits] Images   → train:90,668  val:10,490  test:10,962
[splits] Boxed images → train:715  val:88  test:77
[dataset] train=90,668  val=10,490  test=10,962
[train] Building BaselineModel (resnet50)
Downloading: "https://download.pytorch.org/models/resnet50-0676ba61.pth" to /root/.cache/torch/hub/checkpoints/resnet50-0676ba61.pth


100%|██████████| 97.8M/97.8M [00:00<00:00, 179MB/s]
/content/dl-project/dl-project-code/src/train.py:241: FutureWarning: `torch.cuda.amp.GradScaler(args...)` is deprecated. Please use `torch.amp.GradScaler('cuda', args...)` instead.
  scaler    = GradScaler(enabled=cfg.get("mixed_precision", False) and device.type == "cuda")


[correlation] 21 significant (i,j) pairs (threshold=0.2)
[correlation] 21 significant (i,j) pairs (threshold=0.2)
[train] Starting run: resnet50_baseline
[train] Epochs=30  batch=32  lr=0.0001  device=cuda


/content/dl-project/dl-project-code/src/train.py:97: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with autocast(enabled=use_amp):


  [E000 step  100] cls=0.3059  attn=0.2489  corr=0.0001  total=0.5548
  [E000 step  200] cls=0.3027  attn=0.2422  corr=0.0001  total=0.5449
  [E000 step  300] cls=0.2983  attn=0.2666  corr=0.0001  total=0.5649
  [E000 step  400] cls=0.2946  attn=0.2773  corr=0.0001  total=0.5719
  [E000 step  500] cls=0.2921  attn=0.2808  corr=0.0001  total=0.5729
  [E000 step  600] cls=0.2904  attn=0.2856  corr=0.0001  total=0.5760
  [E000 step  700] cls=0.2882  attn=0.2710  corr=0.0001  total=0.5592
  [E000 step  800] cls=0.2862  attn=0.2645  corr=0.0001  total=0.5507
  [E000 step  900] cls=0.2843  attn=0.2608  corr=0.0001  total=0.5451
  [E000 step 1000] cls=0.2834  attn=0.2556  corr=0.0001  total=0.5391
  [E000 step 1100] cls=0.2824  attn=0.2554  corr=0.0001  total=0.5379
  [E000 step 1200] cls=0.2810  attn=0.2531  corr=0.0001  total=0.5341
  [E000 step 1300] cls=0.2807  attn=0.2504  corr=0.0001  total=0.5312
  [E000 step 1400] cls=0.2797  attn=0.2520  corr=0.0001  total=0.5318
  [E000 step 1500] c

...

/content/dl-project/dl-project-code/src/train.py:143: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with autocast(enabled=use_amp):


[E010] val_AUC=0.8275  train_total=0.4397  val_total=0.4790  [2389s]


/content/dl-project/dl-project-code/src/train.py:97: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with autocast(enabled=use_amp):


  [E011 step  100] cls=0.1931  attn=0.3161  corr=0.0002  total=0.5093


KeyboardInterrupt: 

In [38]:
# Cell 10 — Phase 2: Train DenseNet121 Baseline
cfg['backbone'] = 'densenet121'
print('Training DenseNet121 baseline...')
history_dn121_base = train(cfg, variant=False)
best = max(h['macro_auc'] for h in history_dn121_base['val'])
print(f'DenseNet121 baseline done. Best val macro AUC: {best:.4f}')


Training DenseNet121 baseline...
[train] Loading data...
[splits] ✓ No patient-level leakage detected.
[splits] Patients → train:7,438  val:930  test:930
[splits] Images   → train:19,204  val:2,248  test:2,438
[splits] Boxed images → train:705  val:96  test:79
[dataset] train=19,204  val=2,248  test=2,438
[train] Building BaselineModel (densenet121)
[correlation] 27 significant (i,j) pairs (threshold=0.2)
[correlation] 27 significant (i,j) pairs (threshold=0.2)
[train] Starting run: densenet121_baseline
[train] Epochs=10  batch=64  lr=0.0001  device=cuda


/content/dl-project/dl-project-code/src/train.py:241: FutureWarning: `torch.cuda.amp.GradScaler(args...)` is deprecated. Please use `torch.amp.GradScaler('cuda', args...)` instead.
  scaler    = GradScaler(enabled=cfg.get("mixed_precision", False) and device.type == "cuda")
/content/dl-project/dl-project-code/src/train.py:97: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with autocast(enabled=use_amp):


  [E000 step  100] cls=0.3055  attn=1.0005  corr=0.0002  total=1.3060
  [E000 step  200] cls=0.2961  attn=0.9914  corr=0.0002  total=1.2876
  [E000 step  300] cls=0.2895  attn=1.0016  corr=0.0002  total=1.2912


/content/dl-project/dl-project-code/src/train.py:143: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with autocast(enabled=use_amp):


[E000] val_AUC=0.6948  train_total=1.2912  val_total=1.1502  [487s]
  ↑ New best AUC=0.6948 — saved /content/drive/MyDrive/DL_Project_Outputs/checkpoints/densenet121_baseline_best.pt


...

/content/dl-project/dl-project-code/src/train.py:97: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with autocast(enabled=use_amp):


  [E009 step  100] cls=0.1672  attn=0.9912  corr=0.0003  total=1.1585
  [E009 step  200] cls=0.1674  attn=1.0040  corr=0.0003  total=1.1716
  [E009 step  300] cls=0.1668  attn=0.9915  corr=0.0003  total=1.1585


/content/dl-project/dl-project-code/src/train.py:143: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with autocast(enabled=use_amp):


[E009] val_AUC=0.7753  train_total=1.1585  val_total=1.1610  [482s]
  ↑ New best AUC=0.7753 — saved /content/drive/MyDrive/DL_Project_Outputs/checkpoints/densenet121_baseline_best.pt
[train] Done. Best val AUC=0.7753. Log → /content/drive/MyDrive/DL_Project_Outputs/logs/densenet121_baseline_log.json


TypeError: list indices must be integers or slices, not str

In [39]:
from src.train import train

# EfficientNet-B0 baseline
cfg['backbone'] = 'efficientnet_b0'
print('='*50 + '\nTraining EfficientNet-B0 baseline...')
train(cfg, variant=False)
print('EfficientNet-B0 baseline done.')

# ResNet50 attention
cfg['backbone'] = 'resnet50'
print('='*50 + '\nTraining ResNet50 attention...')
train(cfg, variant=True)
print('ResNet50 attention done.')

# DenseNet121 attention
cfg['backbone'] = 'densenet121'
print('='*50 + '\nTraining DenseNet121 attention...')
train(cfg, variant=True)
print('DenseNet121 attention done.')

# Ablations
cfg['backbone'] = 'resnet50'
cfg['epochs']   = 5
print('='*50 + '\nAblation 1: -L_attn...')
train(cfg, variant=True, no_lattn=True)
print('Ablation 1 done.')

print('Ablation 2: -L_corr...')
train(cfg, variant=True, no_lcorr=True)
print('Ablation 2 done.')

cfg['epochs'] = 10
print('='*50 + '\nALL TRAINING COMPLETE.')

Training EfficientNet-B0 baseline...
[train] Loading data...
[splits] ✓ No patient-level leakage detected.
[splits] Patients → train:7,438  val:930  test:930
[splits] Images   → train:19,204  val:2,248  test:2,438
[splits] Boxed images → train:705  val:96  test:79
[dataset] train=19,204  val=2,248  test=2,438
[train] Building BaselineModel (efficientnet_b0)
Downloading: "https://download.pytorch.org/models/efficientnet_b0_rwightman-7f5810bc.pth" to /root/.cache/torch/hub/checkpoints/efficientnet_b0_rwightman-7f5810bc.pth


100%|██████████| 20.5M/20.5M [00:00<00:00, 90.7MB/s]


[correlation] 27 significant (i,j) pairs (threshold=0.2)
[correlation] 27 significant (i,j) pairs (threshold=0.2)
[train] Starting run: efficientnet_b0_baseline
[train] Epochs=10  batch=64  lr=0.0001  device=cuda


/content/dl-project/dl-project-code/src/train.py:241: FutureWarning: `torch.cuda.amp.GradScaler(args...)` is deprecated. Please use `torch.amp.GradScaler('cuda', args...)` instead.
  scaler    = GradScaler(enabled=cfg.get("mixed_precision", False) and device.type == "cuda")
/content/dl-project/dl-project-code/src/train.py:97: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with autocast(enabled=use_amp):


  [E000 step  100] cls=0.3021  attn=1.0305  corr=0.0001  total=1.3326
  [E000 step  200] cls=0.2942  attn=0.9924  corr=0.0001  total=1.2866
  [E000 step  300] cls=0.2891  attn=0.9951  corr=0.0001  total=1.2842


...

[E003] val_AUC=0.7088  train_total=0.8939  val_total=0.8103  [489s]
  ↑ New best AUC=0.7088 — saved /content/drive/MyDrive/DL_Project_Outputs/checkpoints/resnet50_attention_no_lcorr_best.pt


/content/dl-project/dl-project-code/src/train.py:97: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with autocast(enabled=use_amp):


  [E004 step  100] cls=0.2643  attn=0.5882  corr=0.0000  total=0.8525
  [E004 step  200] cls=0.2622  attn=0.5954  corr=0.0000  total=0.8577
  [E004 step  300] cls=0.2602  attn=0.5971  corr=0.0000  total=0.8573


/content/dl-project/dl-project-code/src/train.py:143: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with autocast(enabled=use_amp):


[E004] val_AUC=0.7277  train_total=0.8573  val_total=0.7989  [479s]
  ↑ New best AUC=0.7277 — saved /content/drive/MyDrive/DL_Project_Outputs/checkpoints/resnet50_attention_no_lcorr_best.pt
[train] Done. Best val AUC=0.7277. Log → /content/drive/MyDrive/DL_Project_Outputs/logs/resnet50_attention_no_lcorr_log.json
Ablation 2 done.
ALL TRAINING COMPLETE.


In [ ]:
# Cell 11 — Phase 2: Train EfficientNet-B0 Baseline
cfg['backbone'] = 'efficientnet_b0'
print('Training EfficientNet-B0 baseline...')
history_eff_base = train(cfg, variant=False)
best = max(h['macro_auc'] for h in history_eff_base['val'])
print(f'EfficientNet-B0 baseline done. Best val macro AUC: {best:.4f}')


---
## Phase 3 — Attention Variants
Full loss: **L_total = L_cls + λ₁·L_attn + λ₂·L_corr**

Running ResNet50 and DenseNet121 (EfficientNet-B0 is baselines-only per proposal).

In [ ]:
# Cell 12 — Phase 3: Train ResNet50 Attention Variant
cfg['backbone'] = 'resnet50'
print('Training ResNet50 attention variant...')
history_r50_attn = train(cfg, variant=True)
best = max(h['macro_auc'] for h in history_r50_attn['val'])
print(f'ResNet50 attention done. Best val macro AUC: {best:.4f}')


In [ ]:
# Cell 13 — Phase 3: Train DenseNet121 Attention Variant
cfg['backbone'] = 'densenet121'
print('Training DenseNet121 attention variant...')
history_dn121_attn = train(cfg, variant=True)
best = max(h['macro_auc'] for h in history_dn121_attn['val'])
print(f'DenseNet121 attention done. Best val macro AUC: {best:.4f}')


---
## Phase 4 — Ablation Study

| Config | λ₁ (L_attn) | λ₂ (L_corr) |
|--------|------------|------------|
| Full model | ✓ | ✓ |
| −L_attn | 0 | ✓ |
| −L_corr | ✓ | 0 |

All ablations: ResNet50 backbone.

In [ ]:
# Cell 14 — Phase 4: Ablations (ResNet50)
cfg['backbone'] = 'resnet50'

print('Ablation 1: ResNet50 attention  -L_attn (lambda1=0)...')
history_no_lattn = train(cfg, variant=True, no_lattn=True)
best = max(h['macro_auc'] for h in history_no_lattn['val'])
print(f'  Done. Best val macro AUC: {best:.4f}\n')

print('Ablation 2: ResNet50 attention  -L_corr (lambda2=0)...')
history_no_lcorr = train(cfg, variant=True, no_lcorr=True)
best = max(h['macro_auc'] for h in history_no_lcorr['val'])
print(f'  Done. Best val macro AUC: {best:.4f}')

print('\nAll ablations complete.')


---
## Evaluation
Loads every checkpoint from Drive and computes:
- Classification: per-class AUC, macro AUC, F1, DeLong significance test
- Localization: attention IoU + pointing game (variants), Grad-CAM IoU (baselines)

Results saved to `DL_Project_Outputs/logs/all_results.json`

In [47]:
import importlib
import torch.serialization as _ts
importlib.reload(_ts)
import torch
torch.load = _ts.load
print('torch.load restored.')

torch.load restored.


In [48]:


import json, os
import numpy as np
import torch

from src.data.splits  import (load_dataframe, build_balanced_subset,
                               patient_level_split, get_class_weights,
                               compute_cooccurrence_matrix, CLASS_NAMES)
from src.data.dataset import build_dataloaders
from src.models.model import build_model
from src.evaluate     import evaluate_model, run_inference
from src.gradcam      import compute_gradcam_batch
from src.metrics      import batch_localization_metrics, delong_test

device = torch.device(cfg['device'])

def disable_inplace_relu(model):
    for m in model.modules():
        if isinstance(m, torch.nn.ReLU) and m.inplace:
            m.inplace = False

def get_test_loader():
    df_ = load_dataframe(cfg['data_dir'])
    if cfg.get('subset_size'):
        df_ = build_balanced_subset(df_, cfg['subset_size'], cfg['random_seed'])
    tr, va, te = patient_level_split(df_, cfg['val_frac'], cfg['test_frac'], cfg['random_seed'])
    cooc_ = compute_cooccurrence_matrix(tr)
    _, _, tl = build_dataloaders(cfg, tr, va, te)
    return tl, cooc_

def load_model(backbone, variant, ckpt_path):
    cfg['backbone'] = backbone
    tl, cooc_ = get_test_loader()
    model = build_model(cfg, cooc_, variant=variant)
    model.load_state_dict(torch.load(ckpt_path, map_location=device, weights_only=False)['model'])
    model.to(device).eval()
    return model, tl

def ckpt(backbone, tag):
    return os.path.join(cfg['checkpoint_dir'], f'{backbone}_{tag}_best.pt')

def jsonify(obj):
    if isinstance(obj, dict):
        return {k: jsonify(v) for k, v in obj.items() if not str(k).startswith('_')}
    if isinstance(obj, (np.float32, np.float64)): return float(obj)
    if isinstance(obj, (np.int32, np.int64)):     return int(obj)
    if isinstance(obj, np.ndarray):               return obj.tolist()
    if isinstance(obj, list):                     return [jsonify(i) for i in obj]
    return obj

all_results = {}

# ── Baselines ────────────────────────────────────────────────
for backbone in ['resnet50', 'densenet121', 'efficientnet_b0']:
    cp = ckpt(backbone, 'baseline')
    print(f'\n{"-"*50}')
    print(f'Evaluating: {backbone} BASELINE')
    if not os.path.exists(cp):
        print(f'  SKIP — checkpoint not found: {cp}'); continue

    model, tl = load_model(backbone, variant=False, ckpt_path=cp)
    res = evaluate_model(model, tl, device, cfg, f'{backbone}_baseline')

    disable_inplace_relu(model)
    gcam_maps, gcam_masks, gcam_boxes = compute_gradcam_batch(
        model, tl, backbone_name=backbone,
        device=str(device), grid_size=cfg['attention_resolution'])
    gcam_loc = batch_localization_metrics(gcam_maps, gcam_masks, gcam_boxes)

    key = f'{backbone}_baseline'
    all_results[key] = {
        'classification': {k: v for k, v in res.items() if not str(k).startswith('_')},
        'gradcam_loc': gcam_loc,
        '_probs':  res.get('_probs'),
        '_labels': res.get('_labels'),
    }
    print(f'  Macro AUC={res["macro_auc"]:.4f}  GradCAM IoU={gcam_loc["mean_iou"]:.4f}')

# ── Attention variants ───────────────────────────────────────
for backbone in ['resnet50', 'densenet121']:
    cp = ckpt(backbone, 'attention')
    print(f'\n{"-"*50}')
    print(f'Evaluating: {backbone} ATTENTION')
    if not os.path.exists(cp):
        print(f'  SKIP — checkpoint not found: {cp}'); continue

    model, tl = load_model(backbone, variant=True, ckpt_path=cp)
    res = evaluate_model(model, tl, device, cfg, f'{backbone}_attention')

    delong_out = {}
    bk = f'{backbone}_baseline'
    if bk in all_results and all_results[bk].get('_probs') is not None:
        bp, bl = all_results[bk]['_probs'], all_results[bk]['_labels']
        vp, vl = res['_probs'], res['_labels']
        for i, cls_name in enumerate(CLASS_NAMES):
            if vl[:, i].sum() < 2: continue
            z, p = delong_test(vl[:, i], vp[:, i], bp[:, i])
            delong_out[cls_name] = {'z': float(z), 'p': float(p)}

    all_results[f'{backbone}_attention'] = {
        'classification': {k: v for k, v in res.items() if not str(k).startswith('_')},
        'attn_loc': res.get('localization', {}),
        'delong_vs_base': delong_out,
    }
    print(f'  Macro AUC={res["macro_auc"]:.4f}  Attn IoU={res.get("localization",{}).get("mean_iou",0):.4f}')

# ── Ablations ────────────────────────────────────────────────
for tag, label in [('attention_no_lattn', '-L_attn'), ('attention_no_lcorr', '-L_corr')]:
    cp = ckpt('resnet50', tag)
    print(f'\n{"-"*50}')
    print(f'Evaluating ablation: resnet50 {label}')
    if not os.path.exists(cp):
        print(f'  SKIP — checkpoint not found: {cp}'); continue

    model, tl = load_model('resnet50', variant=True, ckpt_path=cp)
    res = evaluate_model(model, tl, device, cfg, f'resnet50_{tag}')
    all_results[f'resnet50_{tag}'] = {
        'classification': {k: v for k, v in res.items() if not str(k).startswith('_')},
        'attn_loc': res.get('localization', {}),
    }
    print(f'  Macro AUC={res["macro_auc"]:.4f}  Attn IoU={res.get("localization",{}).get("mean_iou",0):.4f}')

# ── Save ─────────────────────────────────────────────────────
out_path = os.path.join(cfg['log_dir'], 'all_results.json')
with open(out_path, 'w') as f:
    json.dump(jsonify(all_results), f, indent=2)
print(f'\nAll results saved: {out_path}')
print('Models evaluated:', list(all_results))


--------------------------------------------------
Evaluating: resnet50 BASELINE
[splits] ✓ No patient-level leakage detected.
[splits] Patients → train:7,438  val:930  test:930
[splits] Images   → train:19,204  val:2,248  test:2,438
[splits] Boxed images → train:705  val:96  test:79
[dataset] train=19,204  val=2,248  test=2,438
[correlation] 27 significant (i,j) pairs (threshold=0.2)

[evaluate] Running inference for 'resnet50_baseline'...


/content/dl-project/dl-project-code/src/evaluate.py:66: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with autocast(enabled=use_amp):


[resnet50_baseline] Macro AUC = 0.8434
[resnet50_baseline] Per-class AUC:
    Atelectasis           : 0.795
    Cardiomegaly          : 0.945
    Effusion              : 0.818
    Infiltration          : 0.692
    Mass                  : 0.858
    Nodule                : 0.725
    Pneumonia             : 0.796
    Pneumothorax          : 0.871
    Consolidation         : 0.763
    Edema                 : 0.926
    Emphysema             : 0.947
    Fibrosis              : 0.851
    Pleural_Thickening    : 0.821
    Hernia                : 1.000
[resnet50_baseline] Localisation (supervised attn): {'mean_iou': 0.0, 'pointing_game_acc': 0.0, 'n_boxed': 79}
  Macro AUC=0.8434  GradCAM IoU=0.2220

--------------------------------------------------
Evaluating: densenet121 BASELINE
[splits] ✓ No patient-level leakage detected.
[splits] Patients → train:7,438  val:930  test:930
[splits] Images   → train:19,204  val:2,248  test:2,438
[splits] Boxed images → train:705  val:96  test:79
[dataset] t

/content/dl-project/dl-project-code/src/evaluate.py:66: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with autocast(enabled=use_amp):


[densenet121_baseline] Macro AUC = 0.7911
[densenet121_baseline] Per-class AUC:
    Atelectasis           : 0.764
    Cardiomegaly          : 0.904
    Effusion              : 0.799
    Infiltration          : 0.675
    Mass                  : 0.773
    Nodule                : 0.698
    Pneumonia             : 0.681
    Pneumothorax          : 0.834
    Consolidation         : 0.725
    Edema                 : 0.889
    Emphysema             : 0.905
    Fibrosis              : 0.823
    Pleural_Thickening    : 0.760
    Hernia                : 0.847
[densenet121_baseline] Localisation (supervised attn): {'mean_iou': 0.0, 'pointing_game_acc': 0.0, 'n_boxed': 79}
  Macro AUC=0.7911  GradCAM IoU=0.3120

--------------------------------------------------
Evaluating: efficientnet_b0 BASELINE
[splits] ✓ No patient-level leakage detected.
[splits] Patients → train:7,438  val:930  test:930
[splits] Images   → train:19,204  val:2,248  test:2,438
[splits] Boxed images → train:705  val:96  test:7

...

[densenet121_attention] Macro AUC = 0.7475
[densenet121_attention] Per-class AUC:
    Atelectasis           : 0.705
    Cardiomegaly          : 0.850
    Effusion              : 0.741
    Infiltration          : 0.661
    Mass                  : 0.698
    Nodule                : 0.632
    Pneumonia             : 0.646
    Pneumothorax          : 0.748
    Consolidation         : 0.705
    Edema                 : 0.872
    Emphysema             : 0.786
    Fibrosis              : 0.789
    Pleural_Thickening    : 0.690
    Hernia                : 0.941
[densenet121_attention] Localisation (supervised attn): {'mean_iou': 0.31784122136436277, 'pointing_game_acc': 0.46835443037974683, 'n_boxed': 79}
  Macro AUC=0.7475  Attn IoU=0.3178

--------------------------------------------------
Evaluating ablation: resnet50 -L_attn
[splits] ✓ No patient-level leakage detected.
[splits] Patients → train:7,438  val:930  test:930
[splits] Images   → train:19,204  val:2,248  test:2,438
[splits] Boxed i

/content/dl-project/dl-project-code/src/evaluate.py:66: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with autocast(enabled=use_amp):


[resnet50_attention_no_lattn] Macro AUC = 0.8000
[resnet50_attention_no_lattn] Per-class AUC:
    Atelectasis           : 0.766
    Cardiomegaly          : 0.906
    Effusion              : 0.800
    Infiltration          : 0.685
    Mass                  : 0.777
    Nodule                : 0.702
    Pneumonia             : 0.706
    Pneumothorax          : 0.845
    Consolidation         : 0.737
    Edema                 : 0.890
    Emphysema             : 0.912
    Fibrosis              : 0.830
    Pleural_Thickening    : 0.743
    Hernia                : 0.902
[resnet50_attention_no_lattn] Localisation (supervised attn): {'mean_iou': 0.14693906340317908, 'pointing_game_acc': 0.2911392405063291, 'n_boxed': 79}
  Macro AUC=0.8000  Attn IoU=0.1469

--------------------------------------------------
Evaluating ablation: resnet50 -L_corr
[splits] ✓ No patient-level leakage detected.
[splits] Patients → train:7,438  val:930  test:930
[splits] Images   → train:19,204  val:2,248  test:2,438

/content/dl-project/dl-project-code/src/evaluate.py:66: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with autocast(enabled=use_amp):


[resnet50_attention_no_lcorr] Macro AUC = 0.7479
[resnet50_attention_no_lcorr] Per-class AUC:
    Atelectasis           : 0.733
    Cardiomegaly          : 0.857
    Effusion              : 0.725
    Infiltration          : 0.666
    Mass                  : 0.670
    Nodule                : 0.636
    Pneumonia             : 0.649
    Pneumothorax          : 0.779
    Consolidation         : 0.685
    Edema                 : 0.865
    Emphysema             : 0.786
    Fibrosis              : 0.791
    Pleural_Thickening    : 0.684
    Hernia                : 0.943
[resnet50_attention_no_lcorr] Localisation (supervised attn): {'mean_iou': 0.30963667134553213, 'pointing_game_acc': 0.46835443037974683, 'n_boxed': 79}
  Macro AUC=0.7479  Attn IoU=0.3096

All results saved: /content/drive/MyDrive/DL_Project_Outputs/logs/all_results.json
Models evaluated: ['resnet50_baseline', 'densenet121_baseline', 'efficientnet_b0_baseline', 'resnet50_attention', 'densenet121_attention', 'resnet50_attentio

In [57]:
# Cell 16 — Report-ready tables (Tables 2, 3, 4 + Appendix A)
import json, os
import pandas as pd

with open(os.path.join(cfg['log_dir'], 'all_results.json')) as f:
    results = json.load(f)

CLASS_NAMES = [
    'Atelectasis','Cardiomegaly','Effusion','Infiltration',
    'Mass','Nodule','Pneumonia','Pneumothorax',
    'Consolidation','Edema','Emphysema','Fibrosis',
    'Pleural_Thickening','Hernia'
]
MODEL_DISPLAY = {
    'resnet50_baseline'           : 'ResNet50-Base',
    'densenet121_baseline'        : 'DenseNet121-Base',
    'efficientnet_b0_baseline'    : 'EffNet-B0-Base',
    'resnet50_attention'          : 'ResNet50-Attn',
    'densenet121_attention'       : 'DenseNet121-Attn',
    'resnet50_attention_no_lattn' : 'ResNet50 -L_attn',
    'resnet50_attention_no_lcorr' : 'ResNet50 -L_corr',
}

# TABLE 2
print('\n' + '='*68)
print('TABLE 2 — Classification Performance')
print('='*68)
rows = []
for key, label in MODEL_DISPLAY.items():
    if key not in results: continue
    cls = results[key]['classification']
    f1v = list(cls.get('per_class_f1', {}).values())
    mf1 = sum(f1v)/max(1,len(f1v))
    dp  = '—'
    if results[key].get('delong_vs_base'):
        ps = [v['p'] for v in results[key]['delong_vs_base'].values()]
        if ps: dp = f'{min(ps):.4f}'
    rows.append({'Model':label,'Macro AUC':f"{cls.get('macro_auc',0):.4f}",
                 'Macro F1':f'{mf1:.4f}','DeLong p (min)':dp})
print(pd.DataFrame(rows).to_string(index=False))

# TABLE 3
print('\n' + '='*68)
print('TABLE 3 — Localization')
print('='*68)
bnames = {'resnet50':'ResNet50','densenet121':'DenseNet121','efficientnet_b0':'EffNet-B0'}
lrows = []
for bb in ['resnet50','densenet121','efficientnet_b0']:
    bk, ak = f'{bb}_baseline', f'{bb}_attention'
    nm = bnames[bb]
    if bk in results and 'gradcam_loc' in results[bk]:
        g = results[bk]['gradcam_loc']
        lrows.append({'Model':f'{nm}-Base','Method':'Grad-CAM',
                      'Mean IoU':f"{g.get('mean_iou',0):.4f}",
                      'Point. Game':f"{g.get('pointing_game_acc',0):.4f}"})
    if ak in results and 'attn_loc' in results[ak]:
        a = results[ak]['attn_loc']
        d = a.get('mean_iou',0) - results.get(bk,{}).get('gradcam_loc',{}).get('mean_iou',0)
        lrows.append({'Model':f'{nm}-Attn','Method':'Sup. Attn.',
                      'Mean IoU':f"{a.get('mean_iou',0):.4f}",
                      'Point. Game':f"{a.get('pointing_game_acc',0):.4f}",
                      'Delta IoU':f'{d:+.4f}'})
print(pd.DataFrame(lrows).to_string(index=False))

# TABLE 4
print('\n' + '='*68)
print('TABLE 4 — Ablation Study (ResNet50)')
print('='*68)
arows = []
for key, label in [
    ('resnet50_baseline',           'Base (no L_attn, no L_corr)'),
    ('resnet50_attention_no_lcorr', '+L_attn only  (-L_corr)'),
    ('resnet50_attention_no_lattn', '+L_corr only  (-L_attn)'),
    ('resnet50_attention',          'Full model    (+L_attn +L_corr)'),
]:
    if key not in results: continue
    cls = results[key]['classification']
    loc = results[key].get('attn_loc', results[key].get('gradcam_loc', {}))
    arows.append({'Config':label,'Macro AUC':f"{cls.get('macro_auc',0):.4f}",
                  'Mean IoU':f"{loc.get('mean_iou',0):.4f}",
                  'Point. Game':f"{loc.get('pointing_game_acc',0):.4f}"})
print(pd.DataFrame(arows).to_string(index=False))

# APPENDIX A
print('\n' + '='*68)
print('APPENDIX A — Per-Class AUC')
print('='*68)
cols = [('resnet50_baseline','R50-Base'),('densenet121_baseline','DN121-Base'),
        ('efficientnet_b0_baseline','EffNet'),('resnet50_attention','R50-Attn'),
        ('densenet121_attention','DN121-Attn')]
frows = []
for cn in CLASS_NAMES:
    row = {'Finding': cn}
    for key, lbl in cols:
        auc_d = results.get(key,{}).get('classification',{}).get('per_class_auc',{})
        row[lbl] = f"{auc_d.get(cn,0):.3f}" if key in results else '—'
    frows.append(row)
macro = {'Finding':'Macro'}
for key, lbl in cols:
    macro[lbl] = f"{results.get(key,{}).get('classification',{}).get('macro_auc',0):.3f}" if key in results else '—'
frows.append(macro)
print(pd.DataFrame(frows).to_string(index=False))
print('\n-> Copy these into your Word document.')



TABLE 2 — Classification Performance
           Model Macro AUC Macro F1 DeLong p (min)
   ResNet50-Base    0.8434   0.4647              —
DenseNet121-Base    0.7911   0.4123              —
  EffNet-B0-Base    0.7920   0.3892              —
   ResNet50-Attn    0.7603   0.3459         0.0000
DenseNet121-Attn    0.7475   0.3223         0.0000
ResNet50 -L_attn    0.8000   0.3993              —
ResNet50 -L_corr    0.7479   0.3222              —

TABLE 3 — Localization
           Model     Method Mean IoU Point. Game Delta IoU
   ResNet50-Base   Grad-CAM   0.2220      0.4810       NaN
   ResNet50-Attn Sup. Attn.   0.2991      0.5063   +0.0771
DenseNet121-Base   Grad-CAM   0.3120      0.5570       NaN
DenseNet121-Attn Sup. Attn.   0.3178      0.4684   +0.0058
  EffNet-B0-Base   Grad-CAM   0.2240      0.4430       NaN

TABLE 4 — Ablation Study (ResNet50)
                         Config Macro AUC Mean IoU Point. Game
    Base (no L_attn, no L_corr)    0.8434   0.2220      0.4810
        +L_at

In [50]:
# Cell 17 — Generate and save all figures to Drive
import json, os, glob
from src.plots import (
    plot_training_curves, plot_roc_curves,
    plot_localization_comparison, plot_auc_comparison_table,
)

save_dir = cfg['figure_dir']
log_dir  = cfg['log_dir']

# Training curves for every log file
log_files = sorted(glob.glob(os.path.join(log_dir, '*_log.json')))
print(f'Training logs found: {len(log_files)}')
for lf in log_files:
    print(f'  Plotting: {os.path.basename(lf)}')
    plot_training_curves(lf, save_dir)

# AUC comparison + localization
res_path = os.path.join(log_dir, 'all_results.json')
if os.path.exists(res_path):
    with open(res_path) as f:
        all_res = json.load(f)

    CLASS_NAMES = [
        'Atelectasis','Cardiomegaly','Effusion','Infiltration',
        'Mass','Nodule','Pneumonia','Pneumothorax',
        'Consolidation','Edema','Emphysema','Fibrosis',
        'Pleural_Thickening','Hernia'
    ]

    if 'resnet50_attention' in all_res and 'resnet50_baseline' in all_res:
        plot_results = {
            'variant': {
                **all_res['resnet50_attention']['classification'],
                'localization': all_res['resnet50_attention'].get('attn_loc', {}),
            },
            'baseline'   : all_res['resnet50_baseline']['classification'],
            'gradcam_loc': all_res['resnet50_baseline'].get('gradcam_loc', {}),
        }
        plot_localization_comparison(plot_results, save_dir)
        plot_auc_comparison_table(plot_results, CLASS_NAMES, save_dir)

    # ROC curves
    if 'resnet50_attention' in all_res:
        auc_d = all_res['resnet50_attention']['classification'].get('per_class_auc', {})
        if auc_d:
            plot_roc_curves(auc_d, CLASS_NAMES, save_dir)

figures = sorted(glob.glob(os.path.join(save_dir, '*.png')))
print(f'\n{len(figures)} figures saved to: {save_dir}')
for fig in figures:
    print(f'  {os.path.basename(fig)}')


Training logs found: 6
  Plotting: densenet121_attention_log.json
[plots] Training curves → /content/drive/MyDrive/DL_Project_Outputs/figures/densenet121_attention_log_curves.png
  Plotting: densenet121_baseline_log.json
[plots] Training curves → /content/drive/MyDrive/DL_Project_Outputs/figures/densenet121_baseline_log_curves.png
  Plotting: efficientnet_b0_baseline_log.json
[plots] Training curves → /content/drive/MyDrive/DL_Project_Outputs/figures/efficientnet_b0_baseline_log_curves.png
  Plotting: resnet50_attention_log.json
[plots] Training curves → /content/drive/MyDrive/DL_Project_Outputs/figures/resnet50_attention_log_curves.png
  Plotting: resnet50_attention_no_lattn_log.json
[plots] Training curves → /content/drive/MyDrive/DL_Project_Outputs/figures/resnet50_attention_no_lattn_log_curves.png
  Plotting: resnet50_attention_no_lcorr_log.json
[plots] Training curves → /content/drive/MyDrive/DL_Project_Outputs/figures/resnet50_attention_no_lcorr_log_curves.png
[plots] Localisatio

TypeError: plot_roc_curves() missing 1 required positional argument: 'save_dir'

In [55]:
import torch, numpy as np, os
from src.models.model import build_model
from src.plots import plot_roc_curves

CLASS_NAMES = ['Atelectasis','Cardiomegaly','Effusion','Infiltration',
               'Mass','Nodule','Pneumonia','Pneumothorax',
               'Consolidation','Edema','Emphysema','Fibrosis',
               'Pleural_Thickening','Hernia']

device = torch.device(cfg['device'])

# Load best model (ResNet50 attention)
cfg['backbone'] = 'resnet50'
cp = os.path.join(cfg['checkpoint_dir'], 'resnet50_attention_best.pt')
_, cooc_ = get_test_loader()
model = build_model(cfg, cooc_, variant=True)
model.load_state_dict(torch.load(cp, map_location=device, weights_only=False)['model'])
model.to(device).eval()

# Run inference
all_probs, all_labels = [], []
with torch.no_grad():
    for imgs, labels, masks, has_box in test_loader:
        imgs = imgs.to(device)
        out  = model(imgs)
        preds = out[0] if isinstance(out, (list, tuple)) else out
        all_probs.append(torch.sigmoid(preds).cpu().numpy())
        all_labels.append(labels.numpy())

y_score = np.concatenate(all_probs,  axis=0)
y_true  = np.concatenate(all_labels, axis=0)

# Plot ROC curves
plot_roc_curves(y_true, y_score, CLASS_NAMES, cfg['figure_dir'])
print('ROC curves saved to Drive.')

[splits] ✓ No patient-level leakage detected.
[splits] Patients → train:7,438  val:930  test:930
[splits] Images   → train:19,204  val:2,248  test:2,438
[splits] Boxed images → train:705  val:96  test:79
[dataset] train=19,204  val=2,248  test=2,438
[correlation] 27 significant (i,j) pairs (threshold=0.2)
[plots] ROC curves → /content/drive/MyDrive/DL_Project_Outputs/figures/ROC_Curves_roc.png
ROC curves saved to Drive.


In [56]:
# Cell 18 — Final summary
import glob, os

print('='*58)
print('  TRAINING + EVALUATION COMPLETE')
print('='*58)

ckpts   = sorted(glob.glob(os.path.join(cfg['checkpoint_dir'], '*.pt')))
logs    = sorted(glob.glob(os.path.join(cfg['log_dir'], '*.json')))
figures = sorted(glob.glob(os.path.join(cfg['figure_dir'], '*.png')))

print(f'\nCheckpoints ({len(ckpts)}):')
for c in ckpts:
    mb = os.path.getsize(c)/1e6
    print(f'  {os.path.basename(c):52s} {mb:5.0f} MB')

print(f'\nLogs ({len(logs)}):')
for l in logs: print(f'  {os.path.basename(l)}')

print(f'\nFigures ({len(figures)}):')
for fig in figures: print(f'  {os.path.basename(fig)}')

print(f'\nAll outputs: {cfg["output_dir"]}')
print()
print('NEXT STEPS:')
print('  1. Cell 16 -> print all report tables')
print('  2. Copy numbers into DL_Project_Report.docx')
print('  3. Download figures from Drive -> add to report/figures/')
print('  4. Fill in Discussion section with observations')
print('  Done!')


  TRAINING + EVALUATION COMPLETE

Checkpoints (7):
  densenet121_attention_best.pt                           86 MB
  densenet121_baseline_best.pt                            85 MB
  efficientnet_b0_baseline_best.pt                        49 MB
  resnet50_attention_best.pt                             289 MB
  resnet50_attention_no_lattn_best.pt                    289 MB
  resnet50_attention_no_lcorr_best.pt                    289 MB
  resnet50_baseline_best.pt                              283 MB

Logs (7):
  all_results.json
  densenet121_attention_log.json
  densenet121_baseline_log.json
  efficientnet_b0_baseline_log.json
  resnet50_attention_log.json
  resnet50_attention_no_lattn_log.json
  resnet50_attention_no_lcorr_log.json

Figures (9):
  ROC_Curves_roc.png
  auc_comparison.png
  densenet121_attention_log_curves.png
  densenet121_baseline_log_curves.png
  efficientnet_b0_baseline_log_curves.png
  localization_comparison.png
  resnet50_attention_log_curves.png
  resnet50_attention_

In [63]:
import os

# Search local Colab storage
for root, dirs, files in os.walk("/content"):
    for f in files:
        if f.endswith(".png") and f[0].isdigit():
            print("IMAGE_DIR =", root)
            print("Sample file:", f)
            break
    else:
        continue
    break

# Also find src/
for root, dirs, files in os.walk("/content"):
    if "src" in dirs:
        print("CODE_DIR =", root)
        break

IMAGE_DIR = /content/nih-chest-xrays/images
Sample file: 00025934_000.png
CODE_DIR = /content/dl-project/dl-project-code


In [66]:
# Run this first to find the correct layer path
print(type(model.backbone))
print(dir(model.backbone))
# Then check one level deeper:
for name, module in model.backbone.named_children():
    print(name, "->", type(module).__name__)

<class 'src.models.backbone.ResNet50Backbone'>
['T_destination', '__annotations__', '__call__', '__class__', '__delattr__', '__dict__', '__dir__', '__doc__', '__eq__', '__format__', '__ge__', '__getattr__', '__getattribute__', '__getstate__', '__gt__', '__hash__', '__init__', '__init_subclass__', '__le__', '__lt__', '__module__', '__ne__', '__new__', '__reduce__', '__reduce_ex__', '__repr__', '__setattr__', '__setstate__', '__sizeof__', '__str__', '__subclasshook__', '__weakref__', '_apply', '_backward_hooks', '_backward_pre_hooks', '_buffers', '_call_impl', '_compiled_call_impl', '_forward_hooks', '_forward_hooks_always_called', '_forward_hooks_with_kwargs', '_forward_pre_hooks', '_forward_pre_hooks_with_kwargs', '_get_backward_hooks', '_get_backward_pre_hooks', '_get_name', '_is_full_backward_hook', '_load_from_state_dict', '_load_state_dict_post_hooks', '_load_state_dict_pre_hooks', '_maybe_warn_non_full_backward_hook', '_modules', '_named_members', '_non_persistent_buffers_set', '_

In [68]:
import subprocess
result = subprocess.run(['find', '/content', '-name', 'BBox_List_2017.csv', '-type', 'f'],
                       capture_output=True, text=True)
print(result.stdout or "Not found in /content")

result2 = subprocess.run(['find', '/content/drive/MyDrive', '-name', 'BBox_List_2017.csv', '-type', 'f'],
                        capture_output=True, text=True)
print(result2.stdout or "Not found in Drive")

/content/nih-chest-xrays/BBox_List_2017.csv

Not found in Drive


In [73]:
# ============================================================
# PASTE THIS AS A NEW CELL — generates heatmap_overlays.png
# ============================================================
import os, sys, cv2, torch, numpy as np
import matplotlib.pyplot as plt
import matplotlib.patches as patches
import pandas as pd
from pathlib import Path
from PIL import Image
import torchvision.transforms as T

IMAGE_DIR = "/content/nih-chest-xrays/images"
CODE_DIR  = "/content/dl-project/dl-project-code"
DRIVE_OUT = Path("/content/drive/MyDrive/DL_Project_Outputs")
CKPT_PATH = DRIVE_OUT / "checkpoints/resnet50_attention_best.pt"
BBOX_CSV  = Path("/content/nih-chest-xrays/BBox_List_2017.csv")
SAVE_PATH = DRIVE_OUT / "figures/heatmap_overlays.png"
DEVICE    = "cuda" if torch.cuda.is_available() else "cpu"

CLASS_NAMES = [
    "Atelectasis","Cardiomegaly","Effusion","Infiltration","Mass",
    "Nodule","Pneumonia","Pneumothorax","Consolidation","Edema",
    "Emphysema","Fibrosis","Pleural_Thickening","Hernia"
]
TRANSFORM = T.Compose([
    T.Resize((224, 224)), T.ToTensor(),
    T.Normalize([0.485,0.456,0.406],[0.229,0.224,0.225])
])

# ── Fix for PyTorch 2.6 ───────────────────────────────────────
try:
    import numpy as _np
    torch.serialization.add_safe_globals([_np.core.multiarray.scalar])
except Exception:
    pass

# ── Load model ────────────────────────────────────────────────
sys.path.insert(0, CODE_DIR)
from src.models.model import build_model

model = build_model({
    "backbone": "resnet50", "use_attention": True,
    "use_correlation": True, "num_classes": 14, "grid_size": 7
}).to(DEVICE)
ckpt = torch.load(CKPT_PATH, map_location=DEVICE, weights_only=False)
model.load_state_dict(ckpt.get("model_state_dict", ckpt), strict=False)
model.eval()
print("Model loaded on", DEVICE)

# ── Grad-CAM hooks ────────────────────────────────────────────
# backbone.features is a Sequential — hook the last block
grads, acts = {}, {}
target_layer = model.backbone.features[-1]
print("Target layer:", type(target_layer).__name__)

target_layer.register_forward_hook(
    lambda m, i, o: acts.update({"v": o.detach()}))
target_layer.register_full_backward_hook(
    lambda m, gi, go: grads.update({"v": go[0].detach()}))

def get_gradcam(inp, cls_idx):
    model.zero_grad()
    out = model(inp)
    if isinstance(out, dict):
        logits = out["logits"]
    elif isinstance(out, (tuple, list)):
        logits = out[0]
    else:
        logits = out
    logits[0, cls_idx].backward(retain_graph=True)
    g = grads["v"]
    a = acts["v"]
    if g.dim() == 4: g = g.squeeze(0)
    if a.dim() == 4: a = a.squeeze(0)
    w   = g.mean(dim=[1, 2], keepdim=True)
    cam = torch.relu((w * a).sum(0)).cpu().numpy()
    cam = cv2.resize(cam, (224, 224))
    mn, mx = cam.min(), cam.max()
    return (cam - mn) / (mx - mn + 1e-8)

def get_attn(out):
    if isinstance(out, dict) and "attn_map" in out:
        a = out["attn_map"].squeeze().detach().cpu().numpy()
    elif isinstance(out, (tuple, list)) and len(out) > 1:
        a = out[1].squeeze().detach().cpu().numpy()
    else:
        return None
    a = cv2.resize(a, (224, 224))
    mn, mx = a.min(), a.max()
    return (a - mn) / (mx - mn + 1e-8)

# ── Load bbox CSV ─────────────────────────────────────────────
bbox_df = pd.read_csv(BBOX_CSV)
bbox_df.columns = [c.strip() for c in bbox_df.columns]
img_col, finding_col = bbox_df.columns[0], bbox_df.columns[1]
x_col, y_col, w_col, h_col = (bbox_df.columns[2], bbox_df.columns[3],
                               bbox_df.columns[4], bbox_df.columns[5])

# ── Pick up to 6 images (one per unique finding) ──────────────
seen, rows = set(), []
for _, row in bbox_df.iterrows():
    finding  = str(row[finding_col]).strip()
    fname    = str(row[img_col]).strip()
    img_path = Path(IMAGE_DIR) / fname
    if not img_path.exists():
        continue
    if finding not in seen and len(rows) < 6:
        seen.add(finding)
        rows.append({
            "path": img_path, "finding": finding,
            "bx": float(row[x_col]), "by": float(row[y_col]),
            "bw": float(row[w_col]), "bh": float(row[h_col])
        })
print(f"Found {len(rows)} images: {[r['finding'] for r in rows]}")

# ── Build 6×4 figure ──────────────────────────────────────────
n = len(rows)
fig, axes = plt.subplots(n, 4, figsize=(15, 3.6 * n))
if n == 1:
    axes = axes[np.newaxis, :]

for ax, title in zip(axes[0], ["Chest X-Ray", "Supervised Attn (ours)",
                                 "Grad-CAM (baseline)", "Ground-Truth Box"]):
    ax.set_title(title, fontsize=11, fontweight="bold", pad=6)

for ri, r in enumerate(rows):
    pil   = Image.open(r["path"]).convert("RGB")
    ow, oh = pil.size
    inp   = TRANSFORM(pil).unsqueeze(0).to(DEVICE)
    imgr  = np.array(pil.resize((224, 224)))

    bx = r["bx"] * 224 / ow;  by = r["by"] * 224 / oh
    bw = r["bw"] * 224 / ow;  bh = r["bh"] * 224 / oh
    cls_idx = CLASS_NAMES.index(r["finding"]) if r["finding"] in CLASS_NAMES else 0

    with torch.enable_grad():
        inp2 = inp.clone().detach().requires_grad_(True)
        out  = model(inp2)

    attn = get_attn(out)
    gcam = get_gradcam(inp2, cls_idx)

    axs = axes[ri]
    axs[0].imshow(imgr, cmap="gray")
    axs[0].set_ylabel(r["finding"], fontsize=9, rotation=90, va="center", labelpad=4)

    axs[1].imshow(imgr, cmap="gray")
    if attn is not None:
        axs[1].imshow(attn, cmap="Blues", alpha=0.55)
    else:
        axs[1].text(0.5, 0.5, "Attn N/A", transform=axs[1].transAxes,
                    ha="center", va="center", color="red", fontsize=9)

    axs[2].imshow(imgr, cmap="gray")
    axs[2].imshow(gcam, cmap="Reds", alpha=0.55)

    axs[3].imshow(imgr, cmap="gray")
    axs[3].add_patch(patches.Rectangle(
        (bx, by), bw, bh, linewidth=2.5, edgecolor="lime", facecolor="none"))

    for ax in axs:
        ax.axis("off")

plt.suptitle("Supervised Attention vs Grad-CAM vs Ground-Truth Bounding Box",
             fontsize=13, fontweight="bold", y=1.01)
plt.tight_layout()
plt.savefig(SAVE_PATH, dpi=150, bbox_inches="tight")
plt.show()
print(f"\nSaved to: {SAVE_PATH}")
print("Download from Drive and upload here so I can add it to the report.")


Model loaded on cuda
Target layer: Sequential
Found 6 images: ['Atelectasis', 'Cardiomegaly', 'Effusion', 'Infiltrate', 'Mass', 'Nodule']

Saved to: /content/drive/MyDrive/DL_Project_Outputs/figures/heatmap_overlays.png
Download from Drive and upload here so I can add it to the report.
